In [ ]:
!pip install pandas numpy nltk scikit-learn seaborn matplotlib

In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
import matplotlib.pyplot as plt
import seaborn as sns

from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import io

df = pd.read_csv(io.BytesIO(uploaded['flipkart.csv']))
df.head()

In [ ]:
#Create Sentiment Column (IMPORTANT)
# here what we done is loop we hvve give f rating above 4 the it will positive less then it will be negative
def get_sentiment(rating):
    if rating >= 4:
        return "positive"
    else:
        return "negative"

df["sentiment"] = df["Rating"].apply(get_sentiment)
df.head()

In [ ]:
# Text Cleaning
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    text = text.split()
    text = [word for word in text if word not in stop_words]
    return " ".join(text)

df["clean_review"] = df["Review"].apply(clean_text)

In [ ]:
# Data Visualization
sns.countplot(x=df["sentiment"])
plt.title("Sentiment Distribution")
plt.show()

In [ ]:
# TF-IDF Vectorization
tfidf = TfidfVectorizer(max_features=5000)

X = tfidf.fit_transform(df["clean_review"]).toarray()
y = df["sentiment"]

In [ ]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# Model Training
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

In [ ]:
# Predictions
y_pred = model.predict(X_test)

In [ ]:
# Accuracy + Report
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
def predict_sentiment(review):
    review = clean_text(review)
    vector = tfidf.transform([review]).toarray()
    return model.predict(vector)[0]

print(predict_sentiment("This laptop is amazing and fast"))
print(predict_sentiment("Very bad product, waste of money"))